### Architecture

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
import dataclasses
from dataclasses import dataclass
from datasets import load_dataset
from transformers import BertTokenizerFast
import re
import random

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/logitbook_checkpoints", exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [ ]:
@dataclass
class Config:
  vocab_size: int = 30522
  n_layers: int = 4       # 12
  d_model: int = 256      # 768
  n_heads: int = 4        # 12
  d_ff: int = 1024        # 3072 (4 x d_model)
  dropout: float = 0.1
  pad_id: int = 0
  max_seq_len: int = 128
  n_types: int = 2        # segment A / segment B

In [ ]:
class Embeddings(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
    self.pos_embedding = nn.Embedding(config.max_seq_len, config.d_model)
    self.segment_embedding = nn.Embedding(config.n_types, config.d_model)
    self.layer_norm = nn.LayerNorm(config.d_model)
    self.dropout = nn.Dropout(config.dropout)

  def forward(self, x, segment_ids=None):
    B, T = x.size()
    positions = torch.arange(T, device=x.device).unsqueeze(0) # (1, T)

    if segment_ids is None:
      segment_ids = torch.zeros_like(x, dtype=torch.long, device=x.device) # (B, T)

    x = self.token_embedding(x) + self.pos_embedding(positions) + self.segment_embedding(segment_ids) # (B, T, C)
    x = self.layer_norm(x) # (B, T, C)
    x = self.dropout(x) # (B, T, C)
    return x

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.fc1 = nn.Linear(config.d_model, config.d_ff)
    self.fc2 = nn.Linear(config.d_ff, config.d_model)
    self.gelu = nn.GELU()

  def forward(self, x):
    return self.fc2(self.gelu(self.fc1(x))) # (B, T, d_ff) => (B, T, C)

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.d_model = config.d_model
    self.n_heads = config.n_heads
    self.head_size = self.d_model // self.n_heads
    self.W_q = nn.Linear(self.d_model, self.d_model)
    self.W_k = nn.Linear(self.d_model, self.d_model)
    self.W_v = nn.Linear(self.d_model, self.d_model)
    self.W_o = nn.Linear(self.d_model, self.d_model)

  def scaled_dot_product(self, Q, K, V, mask=None):
    B, n_heads, T, head_size = Q.size()
    attn_scores = Q @ K.transpose(-1, -2) # (B, n_heads, T, head_size) @ (B, n_heads, head_size, T) => (B, n_heads, T, T)
    attn_scores = attn_scores / (head_size ** 0.5) # (B, n_heads, T, T)

    if mask is not None:
      attn_scores = attn_scores.masked_fill(mask==0, float("-inf")) # (B, n_heads, T, T)

    attn_probs = torch.softmax(attn_scores, dim=-1) # (B, n_heads, T, T)
    return attn_probs @ V # (B, n_heads, T, T) @ (B, n_heads, T, head_size) => (B, n_heads, T, head_size)

  def combine_heads(self, x):
    B, n_heads, T, head_size = x.size()
    return x.transpose(1, 2).reshape(B, T, -1) # (B, T, C)

  def split_heads(self, x):
    B, T, C = x.size()
    return x.view(B, T, self.n_heads, self.head_size).transpose(1, 2) # (B, n_heads, T, C)

  def forward(self, x, mask):
    Q = self.split_heads(self.W_q(x)) # (B, T, C) @ (C, C) => (B, T, C) => (B, n_heads, T, head_size)
    K = self.split_heads(self.W_k(x)) # (B, T, C) @ (C, C) => (B, T, C) => (B, n_heads, T, head_size)
    V = self.split_heads(self.W_v(x)) # (B, T, C) @ (C, C) => (B, T, C) => (B, n_heads, T, head_size)
    attn_output = self.scaled_dot_product(Q, K, V, mask) # (B, n_heads, T, head_size)
    return self.W_o(self.combine_heads(attn_output)) # (B, n_heads, T, head_size) => (B, T, C) @ (C, C) => (B, T, C)

In [ ]:
class EncoderLayer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.self_attn = MultiHeadAttention(config)
    self.feed_forward = FeedForward(config)
    self.norm1 = nn.LayerNorm(config.d_model)
    self.norm2 = nn.LayerNorm(config.d_model)
    self.dropout = nn.Dropout(config.dropout)

  def forward(self, x, mask=None):
    attn_output = self.self_attn(x, mask)
    x = self.norm1(x + self.dropout(attn_output))
    ff_output = self.feed_forward(x)
    x = self.norm2(x + self.dropout(ff_output))
    return x

In [ ]:
class BERTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embeddings = Embeddings(config)
        self.encoder_layers = nn.ModuleList(EncoderLayer(config) for _ in range(config.n_layers))
        self.pooler_dense = nn.Linear(config.d_model, config.d_model)
        self.pooler_activation = nn.Tanh()

    def forward(self, input_ids, attn_mask=None, segment_ids=None):
      B, T = input_ids.size()

      if attn_mask is None:
        attn_mask = torch.ones(B, T, device=input_ids.device) # (B, T)

      attn_mask = attn_mask.unsqueeze(1).unsqueeze(2) # (B, 1, 1, T)

      x = self.embeddings(input_ids, segment_ids)
      for layer in self.encoder_layers:
          x = layer(x, attn_mask) # (B, T, C)

      cls_token = x[:, 0] # (B, C)
      pooled_output = self.pooler_activation(self.pooler_dense(cls_token)) # (B, C) @ (C, C) => (B, C)
      return x, pooled_output

In [ ]:
class MLMHead(nn.Module):
  def __init__(self, config, token_embedding_weight):
    super().__init__()
    self.dense = nn.Linear(config.d_model, config.d_model)
    self.gelu = nn.GELU()
    self.layer_norm = nn.LayerNorm(config.d_model)
    self.decoder = nn.Linear(config.d_model, config.vocab_size, bias=False)
    self.decoder.weight = token_embedding_weight
    self.bias = nn.Parameter(torch.zeros(config.vocab_size))

  def forward(self, x):
    x = self.gelu(self.dense(x)) # (B, T, C) @ (C, C) => (B, T, C)
    x = self.layer_norm(x) # (B, T, C)
    return self.decoder(x) + self.bias # (B, T, C) @ (C, vocab_size) =>  (B, T, vocab_size) + (vocab_size,) => (B, T, vocab_size)

In [ ]:
class NSPHead(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.classifier = nn.Linear(config.d_model, 2) # isNext / NotNext

  def forward(self, x):
    return self.classifier(x) # (B, C) @ (C, 2) => (B, 2)

In [ ]:
class BERTForPreTraining(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bert = BERTModel(config)
        self.mlm_head = MLMHead(config, self.bert.embeddings.token_embedding.weight)
        self.nsp_head = NSPHead(config)

    def forward(self, input_ids, attention_mask=None, segment_ids=None):
      sequence_output, pooled_output = self.bert(input_ids, attention_mask, segment_ids)
      mlm_logits = self.mlm_head(sequence_output) # (B, T, vocab_size)
      nsp_logits = self.nsp_head(pooled_output) # (B, 2)
      return mlm_logits, nsp_logits

### Training

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [ ]:
VOCAB_SIZE = tokenizer.vocab_size
CLS_ID = tokenizer.cls_token_id
SEP_ID = tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id
MASK_ID = tokenizer.mask_token_id
SPECIAL_IDS = {CLS_ID, SEP_ID, PAD_ID}

print(f"""Vocab size: {VOCAB_SIZE} \nCLS ID: {CLS_ID} \nSEP ID: {SEP_ID} \nPAD ID: {PAD_ID} \nMASK ID: {MASK_ID}""")

Vocab size: 30522 
CLS ID: 101 
SEP ID: 102 
PAD ID: 0 
MASK ID: 103


In [ ]:
raw = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train")

In [ ]:
raw["text"]

Column(['', ' = Valkyria Chronicles III = \n', '', ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game mor

In [ ]:
_sentence_splitter = re.compile(r"(?<=[.!?]) +")

def split_into_sentences(paragraph):
  sentences = _sentence_splitter.split(paragraph.strip())
  return [sentence for sentence in sentences if len(sentence.split()) >= 3]

documents = []
for line in raw["text"]:
  line = line.strip()
  if not line or line.startswith("="):
    continue
  sentences = split_into_sentences(line)
  if len(sentences) >= 2:
    documents.append(sentences)

print("num documents (paragraphs) with >=2 sentences:", len(documents))
print("example document:", documents[0][:3])

num documents (paragraphs) with >=2 sentences: 709144
example document: ['Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit .', 'Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable .', 'Released in January 2011 in Japan , it is the third game in the Valkyria series .']


In [ ]:
def mask_tokens(input_ids, vocab_size, mask_token_id, special_ids, mlm_prob=0.15):
  # input ids for one sequence of shape (T,)
  labels = input_ids.clone()
  input_ids = input_ids.clone()

  special_mask = torch.tensor([int(t.item()) in special_ids for t in input_ids]) # (T,) mask special tokens (True)

  prob_matrix = torch.full(labels.shape, mlm_prob) # (T,) all values are 0.15
  prob_matrix.masked_fill_(special_mask, value=0.0) # (T,) never select [CLS]/[SEP]/[PAD]
  # torch.bernoulli always returns floats (1.0/0.0)
  selected = torch.bernoulli(prob_matrix).bool() # (T,) True = chosen for MLM prediction

  labels[~selected] = -100

  # 80% of selected -> [MASK]
  replace_with_mask = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & selected # (T,)
  input_ids[replace_with_mask] = mask_token_id # (T,) [MASK] token

  # 10% of selected -> random token
  replace_with_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & selected & ~replace_with_mask
  random_tokens = torch.randint(0, vocab_size, labels.shape, dtype=torch.long)
  input_ids[replace_with_random] = random_tokens[replace_with_random]
  return input_ids, labels

In [ ]:
class BERTPretrainingDataset(Dataset):
    def __init__(self, documents, tokenizer, max_seq_len, num_examples):
        self.documents = documents # documents is the entire corpus (list of paras)
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.num_examples = num_examples # num of paras needed (randomly)

    def __len__(self):
        return self.num_examples

    def _get_random_sentence(self, exclude_doc_idx):
      doc_idx = random.randrange(len(self.documents))
      while doc_idx == exclude_doc_idx:
        doc_idx = random.randrange(len(self.documents)) # num of paras
      sentence_idx = random.randrange(len(self.documents[doc_idx]))
      return self.documents[doc_idx][sentence_idx]

    def __getitem__(self, idx):
      doc_idx = random.randrange(len(self.documents))
      doc = self.documents[doc_idx]
      i = random.randrange(len(doc) - 1) # leaves room for a true "next" sentence
      sent_a = doc[i]

      if random.random() < 0.5:
        sent_b = doc[i+1]
        is_next = 1
      else:
        sent_b = self._get_random_sentence(exclude_doc_idx = doc_idx)
        is_next = 0

      budget = self.max_seq_len - 3
      ids_a = self.tokenizer.encode(sent_a, add_special_tokens=False) # don't add special tokens [cls], etc.
      ids_b = self.tokenizer.encode(sent_b, add_special_tokens=False)
      half = budget // 2
      ids_a = ids_a[:half]
      ids_b = ids_b[:budget - len(ids_a)]

      input_ids = [CLS_ID] + ids_a + [SEP_ID] + ids_b + [SEP_ID]
      segment_ids = [0] * (len(ids_a) + 2) + [1] * (len(ids_b) + 1)

      input_ids = torch.tensor(input_ids, dtype=torch.long)
      segment_ids = torch.tensor(segment_ids, dtype=torch.long)

      masked_input_ids, mlm_labels = mask_tokens(input_ids, VOCAB_SIZE, MASK_ID, SPECIAL_IDS)

      return {
          "input_ids": masked_input_ids,
          "segment_ids": segment_ids,
          "mlm_labels": mlm_labels,
          "nsp_label": torch.tensor(is_next, dtype=torch.long)
      }

In [ ]:
def collate_fn(batch):
  max_len = max(item["input_ids"].size(0) for item in batch)

  input_ids = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
  segment_ids = torch.zeros((len(batch), max_len), dtype=torch.long)
  mlm_labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
  attention_mask = torch.zeros((len(batch), max_len), dtype=torch.float)

  nsp_labels = torch.zeros(len(batch), dtype=torch.long)

  for i, item in enumerate(batch):
    L = item["input_ids"].size(0)
    input_ids[i, :L] = item["input_ids"]
    segment_ids[i, :L] = item["segment_ids"]
    mlm_labels[i, :L] = item["mlm_labels"]
    attention_mask[i, :L] = 1.0
    nsp_labels[i] = item["nsp_label"]

  return {
      "input_ids": input_ids,
      "segment_ids": segment_ids,
      "mlm_labels": mlm_labels,
      "attention_mask": attention_mask,
      "nsp_labels": nsp_labels
  }

In [ ]:
config = Config(vocab_size=VOCAB_SIZE, pad_id=PAD_ID)
model = BERTForPreTraining(config).to(device)

print("total params:", sum(p.numel() for p in model.parameters()))

total params: 11169596


In [ ]:
from transformers import get_linear_schedule_with_warmup

NUM_EXAMPLES = 200000
BATCH_SIZE = 64
NUM_EPOCHS = 8
LOG_EVERY = 50
CHECKPOINT_EVERY = 500
CHECKPOINT_PATH = "/content/drive/MyDrive/logitbook_checkpoints/bert_checkpoint.pt"

train_dataset = BERTPretrainingDataset(documents, tokenizer, max_seq_len=128, num_examples=NUM_EXAMPLES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

total_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(0.10 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"total steps: {total_steps}, warmup steps: {warmup_steps}")

total steps: 25000, warmup steps: 2500


In [ ]:
model.train()
for epoch in range(NUM_EPOCHS):
    running_loss, running_mlm, running_nsp, running_nsp_acc = 0.0, 0.0, 0.0, 0.0

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        segment_ids = batch["segment_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        mlm_labels = batch["mlm_labels"].to(device)
        nsp_labels = batch["nsp_labels"].to(device)

        mlm_logits, nsp_logits = model(input_ids, attention_mask, segment_ids)
        mlm_loss = F.cross_entropy(mlm_logits.view(-1, config.vocab_size), mlm_labels.view(-1), ignore_index=-100)
        nsp_loss = F.cross_entropy(nsp_logits, nsp_labels)
        loss = mlm_loss + nsp_loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        running_mlm += mlm_loss.item()
        running_nsp += nsp_loss.item()
        running_nsp_acc += (nsp_logits.argmax(-1) == nsp_labels).float().mean().item()

        if (step + 1) % CHECKPOINT_EVERY == 0:
            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "epoch": epoch,
                "step": step,
                "config": config,
            }, CHECKPOINT_PATH)

    num_steps = len(train_loader)
    print(f"epoch {epoch+1}/{NUM_EPOCHS} "
          f"loss={running_loss/num_steps:.4f} "
          f"mlm={running_mlm/num_steps:.4f} "
          f"nsp={running_nsp/num_steps:.4f} "
          f"nsp_acc={running_nsp_acc/num_steps:.4f} "
          f"lr={scheduler.get_last_lr()[0]:.2e}")

epoch 1/8 loss=22.1788 mlm=21.4836 nsp=0.6952 nsp_acc=0.5006 lr=9.72e-05
epoch 2/8 loss=8.4763 mlm=7.7826 nsp=0.6937 nsp_acc=0.5097 lr=8.33e-05
epoch 3/8 loss=7.4650 mlm=6.7748 nsp=0.6902 nsp_acc=0.5256 lr=6.94e-05
epoch 4/8 loss=7.1066 mlm=6.4225 nsp=0.6841 nsp_acc=0.5472 lr=5.56e-05
epoch 5/8 loss=6.9334 mlm=6.2596 nsp=0.6739 nsp_acc=0.5732 lr=4.17e-05
epoch 6/8 loss=6.8290 mlm=6.1638 nsp=0.6651 nsp_acc=0.5881 lr=2.78e-05
epoch 7/8 loss=6.7677 mlm=6.1089 nsp=0.6588 nsp_acc=0.6009 lr=1.39e-05
epoch 8/8 loss=6.7367 mlm=6.0822 nsp=0.6545 nsp_acc=0.6065 lr=0.00e+00


In [ ]:
tokenizer.save_pretrained("/content/drive/MyDrive/logitbook_checkpoints/tokenizer")

('/content/drive/MyDrive/logitbook_checkpoints/tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/logitbook_checkpoints/tokenizer/tokenizer.json')

In [ ]:
model.eval()

text = "The quick brown fox jumps over the lazy dog"
tokens = tokenizer.tokenize(text)
mask_pos = 3   # index of the word to mask, within `tokens`
masked_tokens = tokens.copy()
masked_tokens[mask_pos] = tokenizer.mask_token

input_ids = torch.tensor([[CLS_ID] + tokenizer.convert_tokens_to_ids(masked_tokens) + [SEP_ID]]).to(device)

with torch.no_grad():
    mlm_logits, _ = model(input_ids)

predicted_id = mlm_logits[0, mask_pos + 1].argmax().item()  # +1 to account for [CLS]
print("original word:", tokens[mask_pos])
print("model's prediction:", tokenizer.convert_ids_to_tokens([predicted_id])[0])

original word: fox
model's prediction: of
